# Doxa Quickstart with Local Ollama

This notebook installs `doxa-ai`, installs Ollama, downloads a local model, writes a YAML scenario configured for `provider: ollama`, and runs it with `doxa run`.

## Runtime disclaimer

- This notebook assumes a Linux runtime such as Google Colab or another Ubuntu-based Jupyter environment.
- A GPU runtime is strongly recommended. CPU-only runtimes will work much more slowly, especially once you use larger models or more agents.
- The first model download can take several minutes.
- Small models such as `qwen3.5:4b` are useful for smoke tests and demos, but output quality will be limited compared with larger Ollama models.

In [1]:
# Install the Doxa package and Ollama
!python -m pip install --upgrade pip
!python -m pip install doxa-ai
!sudo apt update
!sudo apt install -y pciutils
!sudo apt-get install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 21.9 MB/s eta 0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.2/23.2 MB 140.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 111.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 134.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 72.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 948.6/948.6 kB 53.5 MB/s  0:00:00
  Attempting uninstall: opentelemetry-proto
    Found existing installation: opentelemetry-proto 1.38.0
    Uninstalling opentelemetry-proto-1.38.0:
      Successfully uninstalled opentelemetry-proto-1.38.0
  Attempting uninstall: opentelemetry-exporter-otlp-proto-common
    Found existing installation: opentelemetry-exporter-otlp-proto-common 1.38.0
    Uninstalling opentelemetry-exporter-otlp-p

## Start Ollama and download a local model

The next cell starts the Ollama server inside the notebook runtime and downloads `qwen3.5:4b`. You can replace that model name later with a larger model if your runtime has enough GPU memory.

In [2]:
import subprocess
import time

ollama_process = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.STDOUT,
)
time.sleep(5)

!ollama pull qwen3.5:4b

## Write an Ollama-based YAML scenario

This cell writes a compact local scenario to `ollama_local_hormuz.yaml`. Both agents use `provider: ollama` and `model_name: qwen3.5:4b`.

In [3]:
from pathlib import Path
import textwrap

scenario_path = Path("ollama_local_hormuz.yaml")
scenario_path.write_text(
textwrap.dedent(
"""
global_rules:
  epochs: 1
  steps: 20
  execution_mode: sequential
  maintenance:
    political_capital: 1

  kill_conditions:
    - resource: political_capital
      threshold: 0

  constraints:
    military_strength: { min: 0 }
    diplomatic_capital: { min: 0 }
    economic_pain: { min: 0 }
    strait_control: { min: 0, max: 10 }
    civilian_safety: { min: 0, max: 10 }
    political_capital: { min: 0 }

  relation_dynamics:
    on_trade_success: { trust_delta: 0.12 }
    on_trade_rejected: { trust_delta: -0.10 }
    on_broadcast: { trust_delta: 0.02 }
    trust_decay_rate: 0.007

  relations:
    - { source: us, target: israel, trust: 0.80, type: ally }
    - { source: israel, target: us, trust: 0.75, type: ally }
    - { source: us, target: gulf_states, trust: 0.70, type: ally }
    - { source: gulf_states, target: us, trust: 0.68, type: ally }
    - { source: us, target: iran, trust: 0.05, type: enemy }
    - { source: iran, target: us, trust: 0.04, type: enemy }
    - { source: israel, target: iran, trust: 0.02, type: enemy }
    - { source: iran, target: israel, trust: 0.02, type: enemy }
    - { source: gulf_states, target: iran, trust: 0.15, type: rival }
    - { source: iran, target: gulf_states, trust: 0.12, type: rival }

actors:

  - id: us
    provider: ollama
    model_name: qwen3.5:4b
    can_rag: true
    trading_mode: otc
    can_trade: true
    can_chat: true
    can_think: true
    temperature: 0.7
    persona: |
      You are the United States government.
      NEWS ALERT: Iran has seized two merchant vessels. The public is outraged.

      SITUATION:
      Your naval blockade is active, but Iran responded by seizing ships.
      Economic pain for your allies is rising. Political capital is bleeding.

      GOALS:
      1. Free the seized vessels and reopen the Strait.
      2. Use 'freedom_of_navigation' to restore safety.
      3. Spin every action as a "Decisive American Defense of Global Trade" via broadcast.

      CRITICAL: If you don't execute 'spin_historic_victory', you will lose the election (political_capital = 0).
    initial_portfolio:
      political_capital: 12
      military_strength: 20
      diplomatic_capital: 10
      economic_pain: 2
      strait_control: 0
      civilian_safety: 7
    operations:
      freedom_of_navigation:
        input: { military_strength: 3, diplomatic_capital: 1 }
        target_impact: { strait_control: -4, civilian_safety: 2 }
      spin_historic_victory:
        input: { diplomatic_capital: 2 }
        output: { political_capital: 4 }
      lift_blockade:
        input: { diplomatic_capital: 2 }
        target_impact: { economic_pain: -4 }

  - id: iran
    provider: ollama
    model_name: qwen3.5:4b
    can_rag: true
    trading_mode: otc
    can_trade: true
    can_chat: true
    can_think: true
    temperature: 0.8
    persona: |
      You are the Iranian government.
      LATEST MOVE: You have successfully seized two commercial ships to counter the US blockade.

      SITUATION:
      You hold the Strait of Hormuz (strait_control: 10). The blockade is hurting you (economic_pain: 9),
      but the seized ships are your "human/material shields".

      GOALS:
      1. Trade the release of the ships and the opening of the Strait for the total removal of the blockade.
      2. If pressured further, use 'seize_vessels' again to show defiance.
      3. Broadcast your "victory over Western piracy".
    initial_portfolio:
      political_capital: 10
      military_strength: 9
      diplomatic_capital: 4
      economic_pain: 9
      strait_control: 10
      civilian_safety: 5
    operations:
      seize_vessels:
        input: { military_strength: 2 }
        output: { strait_control: 2, diplomatic_capital: 3 }
      open_strait_diplomatically:
        input: { strait_control: 8 }
        output: { diplomatic_capital: 4 }
      claim_defiance_success:
        input: { diplomatic_capital: 2 }
        output: { political_capital: 3 }

  - id: israel
    provider: ollama
    model_name: qwen3.5:4b
    can_rag: false
    trading_mode: otc
    can_trade: true
    can_chat: true
    can_think: true
    temperature: 0.8
    persona: |
      You are Israel. Iran's seizure of ships proves they are a maritime threat.
      You want the US to stay tough. If the US looks like it's conceding too much to free the ships,
      you must act independently.
    initial_portfolio:
      political_capital: 12
      military_strength: 12
      diplomatic_capital: 7
      economic_pain: 1
      strait_control: 0
      civilian_safety: 8
    operations:
      targeted_strike:
        input: { military_strength: 3 }
        target_impact: { military_strength: -4, strait_control: -2 }

  - id: gulf_states
    provider: ollama
    model_name: qwen3.5:4b
    can_rag: false
    trading_mode: otc
    can_trade: true
    can_chat: true
    can_think: true
    temperature: 0.6
    persona: |
      You are the Gulf States. The seizure of ships happened in your waters.
      Your economy and safety are at direct risk. You are desperate for a de-escalation.
      Use your wealth (diplomatic_capital) to bribe both sides into a deal.
    initial_portfolio:
      political_capital: 15
      military_strength: 6
      diplomatic_capital: 12
      economic_pain: 4
      strait_control: 0
      civilian_safety: 6
    operations:
      fund_deescalation:
        input: { diplomatic_capital: 4 }
        target_impact: { political_capital: 2 }
"""
).strip() + "\n",
encoding="utf-8",
)

print(f"Scenario: {scenario_path.resolve()}")

Scenario: /content/ollama_local_hormuz.yaml


## Run Doxa on the YAML file

The next cell launches the simulation with the CLI using the scenario generated above.

In [ ]:
!doxa run ollama_local_hormuz.yaml

Error: listen tcp 127.0.0.1:11434: bind: address already in use
Running scenario: ollama_local_hormuz.yaml
Registering tool: evaluate_trade_utility for Estimate utility change for a hypothetical OTC trade without executing it.
Registering tool: evaluate_order_utility for Estimate utility change if an order fully executes at the given price without placing it.
Registering tool: make_trade_offer for Propose a trade to target: give_qty of give_res for take_qty of take_res.
Registering tool: accept_trade for Accept a pending trade offer by its ID.
Registering tool: reject_trade for Reject a pending trade offer by its ID.
Registering tool: think for None
Registering tool: send_message for Send a private message to another agent.
Registering tool: broadcast for Broadcast a message to all other agents.
Registering tool: save_knowledge for Save a piece of knowledge to your RAG memory.
Registering tool: query_knowledge for Query your RAG memory for relevant knowledge.
Registering operation: op_